# Projet Recherche Opérationnelle — Livrable Check
## Modélisation du problème VRPTW

---

Établissement : CESI — PGE A3 FISA INFO  
Contexte : Réponse à l'appel à manifestation d'intérêt de l'ADEME  
Auteurs : Fayçal Rguig, Rayene Medjtoh, Yanis Bendehane, Victor Schentuleit
Date : 02/04/2026  
Version : 0.1 — Livrable check

---

### Objectif de ce notebook

Ce notebook constitue le livrable check du projet de Recherche Opérationnelle.  
Il présente :

1. La modélisation formelle du problème de tournées de véhicules avec fenêtres temporelles (VRPTW)
2. L'analyse de sa complexité théorique et la démonstration de sa NP-difficulté
3. Le générateur d'instances aléatoires réutilisé par toutes les phases suivantes

Les méthodes de résolution (heuristiques, métaheuristiques, Deep Learning) sont traitées dans le livrable final.

In [ ]:
# ── Imports Globaux ──
import numpy as np #Matrice & calcul numérique
import matplotlib.pyplot as plt #Visualisation
import matplotlib.patches as mpatches #Pour les légendes personnalisées
import networkx as nx #Graphes et algorithmes de graphes
import json #Pour la lecture de fichiers JSON
import time #Pour mesurer le temps d'exécution
import math #Pour les fonctions mathématiques
from itertools import permutations #Pour générer des permutations de listes

# ── Configuration affichage ────S'appliquera a tout les plots──
plt.rcParams['figure.figsize'] = (10, 6) #runtime confguration parameters
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# ── Seed globale pour reproductibilité multi methodes ───
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

print("Environnement chargé.")
print(f"numpy  {np.__version__}")
print(f"networkx  {nx.__version__}")

---

## 1. Contexte et motivation

### 1.1 L'appel à manifestation d'intérêt de l'ADEME

Depuis les années 90, la réduction des émissions de gaz à effet de serre est devenue 
un enjeu mondial majeur. Le protocole de Kyoto (1997), puis les engagements plus 
ambitieux qui ont suivi -- comme la division par 4 des émissions françaises d'ici 2050 
-- ont placé la question de la mobilité au cœur des priorités environnementales.

L'ADEME (Agence de l'Environnement et de la Maîtrise de l'Énergie) a récemment 
lancé un appel à manifestation d'intérêt pour promouvoir des solutions de mobilité 
intelligente adaptées aux territoires. Les applications visées sont nombreuses :

- Distribution du courrier et livraison de colis
- Collecte et traitement des déchets
- Maintenance des équipements urbains (éclairage public, signalisation)
- Transport de personnes en zones peu denses

Notre structure CesiCDP répond à cet appel. L'enjeu est double : proposer 
une solution algorithmique robuste, et démontrer son impact environnemental concret 
par la réduction des kilomètres parcourus et de la consommation de carburant.

### 1.2 Du problème concret au problème algorithmique

Le problème opérationnel est le suivant :

> Chaque jour, des véhicules doivent partir d'un dépôt, livrer un ensemble de 
> clients, puis retourner au dépôt. Chaque client doit être visité exactement une 
> fois, dans un créneau horaire défini, sans dépasser la capacité du véhicule. 
> L'objectif est de minimiser la durée totale des tournées.

Ce problème est connu dans la littérature sous le nom de VRPTW 
(*Vehicle Routing Problem with Time Windows*). Il s'agit d'une extension enrichie 
du célèbre TSP (*Travelling Salesman Problem*), lui-même l'un des problèmes 
les plus étudiés en informatique et en mathématiques depuis les années 1930.

Les deux contraintes que nous retenons pour cette étude sont :

| Contrainte | Description | Justification |
|---|---|---|
| Fenêtres temporelles | Chaque client ne peut être visité qu'entre $a_i$ et $b_i$ | Réalité métier : créneaux de livraison imposés |
| Multi-véhicules + capacité | Plusieurs véhicules disponibles, charge maximale $Q$ | Réalité logistique : flotte limitée en tonnage |

Ces deux contraintes combinées forment le VRPTW, problème de référence 
en recherche opérationnelle, pour lequel des benchmarks standardisés existent 
(instances Solomon, 1987).

### 1.3 Pourquoi modéliser formellement ?

Avant d'écrire le moindre algorithme, il est indispensable de traduire ce problème 
réel en objets mathématiques précis. Cette étape de modélisation remplit trois rôles :

1. Clarifier : une définition formelle évite les ambiguïtés. 
   "Minimiser la durée" peut vouloir dire minimiser la distance totale, 
   le nombre de véhicules, ou le retard cumulé. La modélisation force à choisir.

2. Prouver : la représentation formelle permet de démontrer des propriétés 
   théoriques, notamment la complexité du problème, ce qui justifie les choix 
   algorithmiques qui suivront.

3. Implémenter : un algorithme ne manipule pas des "villes" et des "routes" 
   abstraites, mais des matrices, des vecteurs, des indices. 
   La modélisation est le pont entre la réalité et le code.

Dans la suite de ce notebook, nous construisons pas à pas le modèle mathématique 
complet du VRPTW, puis nous analysons sa complexité théorique, 
et enfin nous implémentons un générateur d'instances pour alimenter 
les phases de résolution suivantes.

---

## 2. Modélisation formelle

### 2.1 Représentation par un graphe

Le réseau routier se modélise naturellement comme un graphe orienté pondéré,
noté $G = (V, A, c)$, où chaque composant traduit un élément du problème réel.

L'ensemble des sommets $V$ représente les lieux à visiter :

$$V = \{0, 1, 2, \ldots, n\}$$

Le sommet $0$ est le dépôt — point de départ et de retour de tous les véhicules.
Les sommets $1$ à $n$ sont les $n$ clients à livrer.

L'ensemble des arcs $A$ représente les trajets possibles entre les lieux :

$$A \subseteq V \times V = \{(i, j) \mid i \in V,\ j \in V,\ i \neq j\}$$

Nous travaillons sur un graphe complet : tout client est directement accessible
depuis n'importe quel autre sommet. Cette hypothèse est réaliste dans un réseau
routier urbain où le chemin le plus court entre deux points peut toujours être calculé.

La fonction de coût $c$ associe à chaque arc $(i, j)$ une valeur réelle positive
représentant la durée de trajet :

$$c : A \rightarrow \mathbb{R}^+, \quad c_{ij} = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}$$

où $(x_i, y_i)$ sont les coordonnées euclidiennes de la ville $i$.
Cette distance satisfait l'inégalité triangulaire :
$c_{ik} \leq c_{ij} + c_{jk}$ pour tout $i, j, k \in V$,
ce qui garantit qu'un trajet direct est toujours au moins aussi court
qu'un trajet avec étape intermédiaire.

In [ ]:
# Illustration : graphe complet sur 5 sommets (1 dépôt + 4 clients)
np.random.seed(GLOBAL_SEED)

n_example = 4  # nombre de clients (hors dépôt)
coords_example = np.random.rand(n_example + 1, 2) * 100

# Construction du graphe complet orienté
G_example = nx.DiGraph()
for i in range(n_example + 1): # Positions 0 à n_example (0 = dépôt, 1..n_example = clients)
    G_example.add_node(i) #Création des sommets

for i in range(n_example + 1): #Création des arcs avec poids (distance euclidienne)
    for j in range(n_example + 1):
        if i != j:
            dist = np.linalg.norm(coords_example[i] - coords_example[j])
            G_example.add_edge(i, j, weight=round(dist, 1)) #Poids = distance arrondie à 1 décimale

# Affichage
pos = {i: coords_example[i] for i in range(n_example + 1)}
node_colors = ["#CA3807" if i == 0 else "#85AFDA" for i in range(n_example + 1)]
node_labels = {0: 'Dépôt'} | {i: f'Client {i}' for i in range(1, n_example + 1)}

fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_networkx_nodes(G_example, pos, node_color=node_colors,
                       node_size=800, ax=ax)
nx.draw_networkx_labels(G_example, pos, labels=node_labels,
                        font_size=9, font_color='black', ax=ax)
nx.draw_networkx_edges(G_example, pos, alpha=0.3, arrows=True,
                       arrowsize=12, ax=ax,
                       connectionstyle='arc3,rad=0.1')

ax.set_title('Graphe complet G = (V, A, c) — dépôt + 4 clients', pad=12)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Sommets : {list(G_example.nodes)}")
print(f"Nombre d'arcs : {G_example.number_of_edges()} (soit n×(n+1) = {n_example}×{n_example+1})")

### 2.2 La matrice des distances

En pratique, les coûts $c_{ij}$ sont stockés sous forme d'une matrice carrée
$C \in \mathbb{R}^{(n+1) \times (n+1)}$ :

$$C = \begin{pmatrix} 0 & c_{01} & c_{02} & \cdots & c_{0n} 
\\ c_{10} & 0 & c_{12} & \cdots & c_{1n} 
\\ \vdots & & \ddots & & \vdots 
\\ \vdots & & & \ddots 
\\ c_{n0} & c_{n1} & \cdots & & 0 \end{pmatrix}$$

La diagonale est nulle ($c_{ii} = 0$, un sommet est à distance 0 de lui-même).
La matrice est symétrique dans le cas euclidien ($c_{ij} = c_{ji}$),
mais cette propriété n'est pas requise par le modèle : on pourrait
avoir des durées différentes selon le sens de circulation.

L'accès à n'importe quelle distance est en temps constant $O(1)$,
ce qui est essentiel pour les algorithmes qui consultent cette matrice
des millions de fois.

In [ ]:
# Calcul vectorisé de la matrice de distances euclidiennes
# coords_example : tableau (n+1, 2) des coordonnées

def compute_distance_matrix(coords):
    """
    Calcule la matrice de distances euclidiennes entre tous les sommets.
    @param coords : np.ndarray de forme (n+1, 2)
    @return : np.ndarray de forme (n+1, n+1) avec les distances
    """
    # Différence entre toutes les paires de coordonnées en une seule opération
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :] # Broadcast simplifier les boucles
    dist = np.sqrt((diff ** 2).sum(axis=2)) # (n+1, n+1) avec les distances euclidiennes
    return dist 

dist_example = compute_distance_matrix(coords_example)

# Affichage formaté
print("Matrice des distances C (arrondie à 1 décimale) :\n")
header = "      " + "  ".join(
    [f"{'Dép':>6}"] + [f"{'C'+str(i):>6}" for i in range(1, n_example + 1)]
)
print(header)
print("  " + "-" * (len(header) - 2))
row_labels = ['Dép'] + [f'C{i}' for i in range(1, n_example + 1)]
for i, label in enumerate(row_labels):
    row = "  ".join([f"{dist_example[i][j]:6.1f}" for j in range(n_example + 1)])
    print(f"{label:>4} | {row}")

print(f"\nDiagonale nulle : {np.all(np.diag(dist_example) == 0)}")
print(f"Matrice symétrique : {np.allclose(dist_example, dist_example.T)}")

### 2.3 Variables de décision

La modélisation du VRPTW relève de la Programmation Linéaire en Nombres Entiers
(PLNE). Elle repose sur deux types de variables de décision : c'est-à-dire
les quantités que le modèle doit déterminer.

Soit $K = \{1, \ldots, m\}$ l'ensemble des $m$ véhicules disponibles.

La première variable encode les décisions de routage :

$$x_{ij}^k \in \{0, 1\} \quad \forall i, j \in V,\ \forall k \in K$$

$x_{ij}^k = 1$ si et seulement si le véhicule $k$ emprunte l'arc $(i \to j)$
dans sa tournée, $0$ sinon. L'ensemble de toutes ces variables binaires
décrit entièrement quelles routes sont empruntées et par quel véhicule.

La seconde variable encode les décisions temporelles :

$$t_i^k \in \mathbb{R}^+ \quad \forall i \in V,\ \forall k \in K$$

$t_i^k$ représente l'heure d'arrivée du véhicule $k$ au client $i$.
Cette variable est indispensable pour modéliser les fenêtres temporelles.
Elle couple les décisions de routage au planning horaire réel :
l'ordre dans lequel un véhicule visite les clients détermine ses heures d'arrivée.

Le nombre total de variables binaires est $|V|^2 \times |K| = (n+1)^2 \times m$,
ce qui croit rapidement avec $n$ — c'est l'une des sources de la difficulté
du problème.

### 2.4 Fonction objectif

L'objectif est de minimiser la somme des coûts de tous les arcs empruntés
par tous les véhicules :

$$\min \sum_{k \in K} \sum_{i \in V} \sum_{j \in V} c_{ij} \cdot x_{ij}^k$$

pour chaque véhicule k, pour chaque ville de départ i, pour chaque ville d'arrivée j — si le véhicule k emprunte l'arc i→j, ajouter le coût de ce trajet. Sommer tout ça.

Le produit $c_{ij} \cdot x_{ij}^k$ vaut $c_{ij}$ si le véhicule $k$ emprunte
l'arc $(i, j)$, et $0$ sinon. La somme triple ne compte donc que les arcs
réellement utilisés dans la solution finale.

Cette formulation minimise la distance totale parcourue par la flotte.
D'autres objectifs sont possibles — minimiser le nombre de véhicules utilisés,
ou minimiser le retard total — mais ils nécessiteraient une reformulation
du modèle. Nous retenons la distance totale pour sa cohérence avec
l'objectif environnemental de l'ADEME.

---

## 3. Contraintes du VRPTW

### 3.1 Du TSP au VRPTW

Le TSP dans sa forme de base cherche une tournée minimale sur un graphe complet.
Notre problème réel impose deux couches de contraintes supplémentaires,
qui le transforment en VRPTW (Vehicle Routing Problem with Time Windows).

La première couche concerne la logistique de la flotte :
chaque véhicule a une capacité maximale $Q$, et plusieurs véhicules
peuvent opérer en parallèle depuis le dépôt.

La seconde couche concerne le temps :
chaque client $i$ ne peut être visité qu'entre une heure d'ouverture $a_i$
et une heure de fermeture $b_i$.

Ces deux contraintes sont justifiées par le contexte ADEME :
les tournées de livraison en milieu urbain sont soumises à des créneaux
imposés par les clients, et les véhicules ont une charge physique limitée.
Leur combinaison produit le VRPTW, problème de référence en recherche
opérationnelle pour lequel des benchmarks standardisés existent
(instances Solomon, 1987).

Pour chaque contrainte, nous présentons la formulation formelle,
une explication en langage naturel, et un exemple numérique concret.

### 3.2 Contrainte C1 — Couverture

$$\sum_{k \in K} \sum_{j \in V} x_{ij}^k = 1 \qquad \forall i \in V \setminus \{0\}$$

Chaque client $i$ (hors dépôt) doit être visité exactement une fois,
par exactement un véhicule. La somme de tous les arcs sortant de $i$
sur tous les véhicules vaut 1.

Cette contrainte interdit deux situations indésirables :
une visite oubliée (somme = 0) et une double visite (somme $\geq 2$).

Exemple numérique avec 3 clients et 2 véhicules :

$$x_{1,2}^1 + x_{1,3}^1 + x_{1,0}^1 + x_{1,2}^2 + x_{1,3}^2 + x_{1,0}^2 = 1$$

Si le véhicule 1 va du client 1 vers le client 3, alors $x_{1,3}^1 = 1$
et tous les autres termes valent 0. La somme vaut bien 1.


### 3.3 Contrainte C2 — Conservation de flux

$$\sum_{i \in V} x_{ij}^k = \sum_{i \in V} x_{ji}^k \qquad \forall j \in V,\ \forall k \in K$$

Pour chaque sommet $j$ et chaque véhicule $k$, le nombre d'arcs entrants
égale le nombre d'arcs sortants. Si un véhicule arrive quelque part,
il doit aussi en repartir.

Cette contrainte garantit que chaque tournée est un cycle fermé.
Elle interdit qu'un véhicule "s'arrête" en cours de route
ou qu'il apparaisse ou disparaisse en un point intermédiaire.

Elle s'applique aussi au dépôt : le véhicule qui part du dépôt (arc sortant)
doit y revenir (arc entrant). C'est la contrainte de retour au dépôt.

Exemple numérique pour le client 2, véhicule 1 :

$$x_{0,2}^1 + x_{1,2}^1 + x_{3,2}^1 = x_{2,0}^1 + x_{2,1}^1 + x_{2,3}^1$$

Si le véhicule 1 arrive au client 2 depuis le client 1
et repart vers le client 3, alors $x_{1,2}^1 = 1$ et $x_{2,3}^1 = 1$,
les autres termes valent 0. L'égalité $1 = 1$ est vérifiée.

### 3.4 Contrainte C3 — Capacité des véhicules

$$\sum_{i \in V} q_i \cdot \left(\sum_{j \in V} x_{ij}^k\right) \leq Q \qquad \forall k \in K$$

La somme des demandes de tous les clients visités par le véhicule $k$
ne peut pas dépasser sa capacité maximale $Q$.

Le terme $\sum_{j \in V} x_{ij}^k$ vaut 1 si le client $i$ est visité
par le véhicule $k$, 0 sinon. Le produit $q_i \cdot (\ldots)$
ne prend donc en compte que les clients effectivement servis.

Lorsque cette contrainte est active, le problème ne peut pas être résolu
par un seul véhicule : il faut répartir les clients entre plusieurs tournées.
C'est ce qui fait du VRPTW un problème fondamentalement différent du TSP.

Exemple numérique avec $Q = 100$ kg :

| Client | Demande $q_i$ | Véhicule 1 |
|--------|--------------|------------|
| 1      | 40 kg        | oui        |
| 2      | 35 kg        | oui        |
| 3      | 50 kg        | non        |

Charge véhicule 1 : $40 + 35 = 75 \leq 100$ — contrainte respectée.
Si on ajoutait le client 3 : $40 + 35 + 50 = 125 > 100$ — infaisable,
il faut un second véhicule pour le client 3.

### 3.5 Contrainte C4 — Fenêtres temporelles

$$a_i \leq t_i^k \leq b_i \qquad \forall i \in V,\ \forall k \in K$$

L'heure d'arrivée $t_i^k$ du véhicule $k$ chez le client $i$
doit se situer dans l'intervalle $[a_i, b_i]$.

Trois situations se présentent selon l'heure d'arrivée effective :

- Si $t_i^k \in [a_i, b_i]$ : la visite est valide, le service commence immédiatement.
- Si $t_i^k < a_i$ : le véhicule arrive trop tôt et attend.
  L'heure de début de service devient $a_i$.
  Ce temps d'attente est autorisé mais s'accumule et retarde les visites suivantes.
- Si $t_i^k > b_i$ : la fenêtre est fermée. En modélisation stricte (hard constraint),
  cette solution est infaisable. En modélisation souple (soft constraint),
  une pénalité proportionnelle au retard est ajoutée à la fonction objectif.

Nous retenons la modélisation souple pour les phases algorithmiques,
ce qui permet aux heuristiques et au recuit simulé d'explorer
des solutions temporairement infaisables :

$$\text{pénalité}_i^k = \lambda \cdot \max(0,\ t_i^k - b_i)$$

où $\lambda$ est un paramètre de pondération à calibrer.